<a id="unit-u.ba48f863134f44dd9feee9c96b0022a2"></a>

<div>
<div>КЕҢЕЙТУ · APACHE DORIS КӨМЕГІМЕН ДЕРЕКТЕР ҚОЙМАСЫН ҚҰРУ</div>
<h1>6–7-модульдердің кеңейтілген зертханалық жұмысы: схема өзгерістері, жүктеу арқылы жою және қайшылықтарды анықтау</h1>
<p>Мақсатты нұсқа — Doris 4.1.3 · Тәуелсіз ext_* зертханалық кестелері</p>
</div>

[6-модуль](../module06-data-quality/course.md) · [7-модуль](../module07-state-changes/course.md) · [Курстың README файлы](../../README.md)

Алдымен 6 және 7-зертханалық жұмыстарды аяқтаңыз: бұл кеңейту 6-зертханалық жұмыстың <code>orders_clean</code> кестесін оқып, 7-зертханалық жұмыстың нұсқа және оқиға ұғымдарын қолданады. Баптау кеңейту кестелері қайта жасалмай тұрып кірісті тексереді; ол жоқ не толық болмаса, сол дерекқордағы 6-зертханалық жұмысқа сілтемемен өтіп, содан кейін оралыңыз. Кеңейту тек `ext_schema`, `ext_delete_sign` және `ext_event_stage` кестелерін қайта жасайды, <code>orders_clean</code>, `orders_current` және бизнес тарихын өзгертпейді.
25–35 минут бөліңіз. Курс сынақ ортасында бір уақытта бір ядрода орындап, ақауларды анықтау үшін нәтижелерді сақтаңыз. Баптау ұяшығы сынақ ортасының BE-сін көрсетеді: `Alive` true және `Version` `doris-4.1.3-rc02-7126cf65d96`.

Қосылған бағанда ескі жолдар NULL көрсетсе, бағандарын атайтын кейінгі сұрау өзгермесе, ескі нұсқа жойылған тапсырысты қайтармаса және бір оқиға ID-імен келген басқа мазмұн анықталса, жұмыс аяқталады. Бұл кеңейту нақты binlog-тан қалпына келтірмейді және қатар тұтынушылардың қайшылықтарын атомарлы түрде кері қайтаратын қызмет емес.


In [ ]:
from pathlib import Path
import sys
COURSE_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "dw_course").is_dir())
if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))
from dw_course.docker_runtime import connect_sandbox
from dw_course.runtime import expect, fixture, normalized
from dw_course.ui import show_sql, show_response
lab = connect_sandbox(module="extension67")
lab.sql("SHOW BACKENDS", title="The course sandbox's BE", columns=["Host", "Alive", "Version"]);

<a id="unit-u.f024e7f730ba41fc86781aa70fda1321"></a>

<a id="1-adding-columns-compatibility-and-asynchronous-change-status"></a>

## 1. Баған қосу, үйлесімділік және асинхронды өзгеріс күйі

Он тапсырысты тәуелсіз кестеге көшіріп, алдымен кейінгі сұраудың айқын баған проекциясын тіркеңіз. NULL қабылдайтын арна бағанын қосыңыз; ескі жазбаларда NULL болады.
Содан кейін amount_cents түрін INT-тен BIGINT-ке өзгертіп, SHOW ALTER TABLE COLUMN тапсырмасының аяқталуын күтіңіз және нәтижелерді тексеріңіз.
Жеңіл не ауыр болуы операция мен мақсатты нұсқаға байланысты; осы орындаудың ұзақтығынан барлық Schema Change тек метадеректерді өзгертеді деп қорытпаңыз.


In [ ]:
import time
lab.execute("DROP TABLE IF EXISTS ext_schema")
lab.execute('''CREATE TABLE ext_schema (order_id BIGINT, amount_cents INT)
DUPLICATE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1 PROPERTIES("replication_num"="1")''')
lab.execute("INSERT INTO ext_schema SELECT order_id,CAST(order_amount*100 AS INT) FROM orders_clean")
expected = lab.query("SELECT order_id,amount_cents FROM ext_schema ORDER BY order_id")
expect(lab.query("SELECT COUNT(*),SUM(amount_cents) FROM ext_schema"), [(10,140000)])
for statement, column, target_type in (("ALTER TABLE ext_schema ADD COLUMN order_channel VARCHAR(20) NULL", "order_channel", "varchar"),
                                      ("ALTER TABLE ext_schema MODIFY COLUMN amount_cents BIGINT", "amount_cents", "bigint")):
    show_sql("Schema Change",statement)
    lab.execute(statement)
    deadline = time.monotonic()+120
    while True:
        with lab.connection.cursor() as cursor:
            cursor.execute("SHOW ALTER TABLE COLUMN WHERE TableName = 'ext_schema' ORDER BY CreateTime DESC LIMIT 1")
            fields = [c[0] for c in cursor.description]
            jobs = [dict(zip(fields,row)) for row in cursor.fetchall()]
        types = {row[0]: row[1].lower() for row in lab.query("DESC ext_schema")}
        if types.get(column, "").startswith(target_type) and (not jobs or jobs[0]["State"] == "FINISHED"):
            break
        if jobs and jobs[0]["State"] == "CANCELLED":
            raise RuntimeError(str(jobs))
        if time.monotonic()>=deadline:
            raise TimeoutError(str(jobs))
        time.sleep(1)
    lab.sql("SHOW ALTER TABLE COLUMN WHERE TableName = 'ext_schema'")
    expect(lab.query("SELECT order_id,amount_cents FROM ext_schema ORDER BY order_id"), expected)
expect(lab.query("SELECT COUNT(*),COUNT(order_channel) FROM ext_schema"), [(10,0)])
lab.sql("DESC ext_schema");

<a id="unit-u.c8ede2070aa347688ccc6acd5fac9c0d"></a>

<a id="2-load-deletion-markers-and-old-version-replay"></a>

## 2. Жою белгілерін жүктеу және ескі нұсқаны қайта ойнату

Unique Key + Sequence бар тәуелсіз үш бағандық кесте қолданыңыз. Сұрау MERGE және DELETE ON шарты арқылы операция түрлерін жою белгілеріне сәйкестендіреді.
Алдымен CREATED v1 жүктеңіз, содан кейін v2-де жойыңыз, v1-ді қайта ойнатыңыз және соңында жаңа PAID v3 нұсқасын айқын жазыңыз.
Тексеру реті — 1 → 0 → 0 → 1 жол; жаңа нұсқамен қайта құру — ескі жазбаның кездейсоқ қайта тірілуі емес, айқын жаңа оқиға.
Кәдімгі сұрауларға көрінбеу дисктің бірден босауын білдірмейді; мұнда бизнестің қайтарым кестесі өзгермейді.


In [ ]:
import os
import requests
from uuid import uuid4
lab.execute("DROP TABLE IF EXISTS ext_delete_sign")
lab.execute('''CREATE TABLE ext_delete_sign (
order_id BIGINT NOT NULL,status VARCHAR(20) NOT NULL,event_version BIGINT NOT NULL
) UNIQUE KEY(order_id) DISTRIBUTED BY HASH(order_id) BUCKETS 1
PROPERTIES("replication_num"="1","enable_unique_key_merge_on_write"="true","function_column.sequence_col"="event_version")''')
endpoint = os.environ["DW_BE_HTTP_URL"].rstrip("/")
for payload, expected in (("900001,CREATED,1,UPSERT\n",[(900001,"CREATED",1)]),
                          ("900001,CREATED,2,DELETE\n",[]),
                          ("900001,CREATED,1,UPSERT\n",[]),
                          ("900001,PAID,3,UPSERT\n",[(900001,"PAID",3)])):
    headers = {"label":"ext_delete_"+uuid4().hex,"format":"csv","column_separator":",",
               "columns":"order_id,status,event_version,op","merge_type":"MERGE","delete":"op='DELETE'",
               "group_commit":"off_mode","strict_mode":"true","max_filter_ratio":"0"}
    response = requests.put(f"{endpoint}/api/{lab.database}/ext_delete_sign/_stream_load",
                            auth=(lab.user,lab.password),headers=headers,data=payload.encode(),
                            allow_redirects=False,timeout=120)
    response.raise_for_status()
    result = response.json()
    show_response(result, title="Delete-sign Stream Load response")
    expect(result["Status"],"Success")
    expect(lab.query("SELECT order_id,status,event_version FROM ext_delete_sign ORDER BY order_id"),expected)

<a id="unit-u.cc143db17095400aa89da046d8896f22"></a>

<a id="3-the-same-event-id-but-different-content"></a>

## 3. Бірдей оқиға ID-і, бірақ басқа мазмұн

Unique Key(event_id) бірдей ID-і бар жазбаларды ауыстыра алады, бірақ мазмұн қайшылықтарын автоматты анықтамайды. Алдымен бастапқы үміткер оқиғаларды тәуелсіз аралық кестеге салыңыз.
Бірдей мазмұн идемпотенттілік келісімшартына әлі сай; бір ID-дің басқа мазмұны өңдеуді тоқтатуы керек, оны негізгі тарихқа не ағымдағы кестеге тікелей жазуға болмайды.
Бұл мысал тек үш бизнес өрісін сақтайды: order_id/status/version. Өндірістік ережелер толық қалыптандырылған бизнес мазмұнын қамтуы керек.
Бір жазушының бұл «тексеру, содан кейін жазу» көрсетілімі көп жазушыға транзакциялық кепілдік бермейді.


In [ ]:
from dw_course.runtime import expected_failure
lab.execute("DROP TABLE IF EXISTS ext_event_stage")
lab.execute('''CREATE TABLE ext_event_stage (
event_id VARCHAR(32),order_id BIGINT,status VARCHAR(20),event_version BIGINT
) DUPLICATE KEY(event_id) DISTRIBUTED BY HASH(event_id) BUCKETS 1 PROPERTIES("replication_num"="1")''')
lab.execute("INSERT INTO ext_event_stage VALUES ('CONFLICT_DEMO',900001,'PAID',2),('CONFLICT_DEMO',900001,'PAID',2)")
conflicts = """SELECT event_id FROM (
SELECT DISTINCT event_id,order_id,status,event_version FROM ext_event_stage
) v GROUP BY event_id HAVING COUNT(*)>1 ORDER BY event_id"""
expect(lab.query(conflicts),[])
lab.execute("INSERT INTO ext_event_stage VALUES ('CONFLICT_DEMO',900001,'CANCELLED',2)")
with expected_failure("Event content conflict", "Different content for the same ID detected; stop subsequent business writes"):
    expect(lab.query(conflicts),[])
expect(lab.query(conflicts),[("CONFLICT_DEMO",)])
lab.sql("SELECT * FROM ext_event_stage ORDER BY event_id,status");

<a id="unit-u.ecc5657b0d6c4cdc9d0516ab717589cb"></a>

<a id="4-independent-explanation-and-troubleshooting"></a>

## 4. Өз бетінше түсіндіру және ақауларды анықтау

Арна бағаны неге NULL бола алатынын және жаңа баған кейінгі қай SELECT * операторларына не орын бойынша жазуларға әсер ететінін түсіндіріңіз.
2-жою нұсқасы неге ескі 1-нұсқаны бөгейтінін және айқын жаңа 3-нұсқа неге қайта көріне алатынын түсіндіріңіз.
Аралық жазбада тек тапсырыс ID-ін өзгертіп көріп, қайшылықты анықтау қатені неге әлі табатынын түсіндіріңіз.
Schema Change сәтсіздігінде SHOW ALTER TABLE COLUMN; жүктеу сәтсіздігінде жауап пен ErrorURL тексеріңіз; қайшылықтарда дәлелді ауыстырып жоғалтпай, барлық үміткер жазбаны сақтаңыз.

Дереккөздер: [Schema Change](https://doris.apache.org/docs/4.x/table-design/schema-change/),
[Жаңартулар және жою](https://doris.apache.org/docs/4.x/data-operate/update/update-overview/).


In [ ]:
lab.close()